In [1]:
import pickle
from PIL import Image
import torchvision.transforms as T
from io import BytesIO
from PIL import Image, ImageEnhance, ImageOps
import os
import torch
import contextlib
import json
from mylib import yolo_patch_softmax
from mylib import utils
import math
import numpy as np
from pycocotools.coco import COCO
import matplotlib.pyplot as plt
import skimage.io as io
from ultralytics import YOLO 
from dotenv import load_dotenv
from tqdm import tqdm  # For progress tracking
import time
load_dotenv()  # This loads from .env in the current directory

True

In [2]:
# === Load YOLO model ===
yolo_model = YOLO("yolo11x.pt")  # Swap with yolov8s.pt or yolov8x.pt as needed

In [3]:
# Initialize cuda:0 device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
yolo_model = yolo_model.to(device)

Using device: cuda:0


In [4]:
# Load annotations for both datasets
cocos = {
    "train2017": COCO(os.path.join(os.getenv("COCO_DATA"), "annotations/instances_train2017.json")),
    "val2017": COCO(os.path.join(os.getenv("COCO_DATA"), "annotations/instances_val2017.json")),
}

loading annotations into memory...
Done (t=17.50s)
creating index...
index created!
loading annotations into memory...
Done (t=0.61s)
creating index...
index created!


In [5]:
# YOLO ID -- Class mapping from json file
with open('indoor_objects.json') as f:
    indoor_objects = json.load(f)
    indoor_objects = {k: v for d in indoor_objects['indoor_classes'] for k, v in d.items()}

# Bin range -- Class from json file
with open("out_bins_per_class.json") as f:
    out_bins_per_class = json.load(f)

# YOLO IDs -- COCO IDs mapping
train_coco = list(cocos.values())[0]
all_cats = train_coco.loadCats(train_coco.getCatIds())
yolo_id_to_coco_id = {i: cat["id"] for i, cat in enumerate(sorted(all_cats, key=lambda x: x["id"]))} # eg. Yolo ID 79 -> COCO ID 90

# Build lists 
yolo_ids = [int(k) for k in indoor_objects.keys()]
coco_ids = [yolo_id_to_coco_id[int(k)] for k in indoor_objects.keys()]
classes_names = [v for v in indoor_objects.values()]
classes_bins = [out_bins_per_class[v] for v in indoor_objects.values()]
num_classes = len(classes_bins)

# Print information
print("Number of classes:", num_classes,'\n')
print("YOLO IDs:", yolo_ids)
print("COCO IDs:", coco_ids)
print("Classes names:", classes_names)
print("Classes bins:", classes_bins)


Number of classes: 27 

YOLO IDs: [26, 39, 40, 41, 42, 43, 44, 45, 47, 56, 57, 58, 59, 60, 61, 62, 63, 65, 67, 68, 69, 70, 71, 73, 74, 75, 77]
COCO IDs: [31, 44, 46, 47, 48, 49, 50, 51, 53, 62, 63, 64, 65, 67, 70, 72, 73, 75, 77, 78, 79, 80, 81, 84, 85, 86, 88]
Classes names: ['handbag', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'apple', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'remote', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'book', 'clock', 'vase', 'teddy bear']
Classes bins: [[0.0, 0.05, 0.11, 0.19, 0.33, 0.53, 0.84, 1.41, 2.44, 4.87, 99.4], [0.0, 0.06, 0.11, 0.17, 0.25, 0.37, 0.55, 0.87, 1.52, 3.39, 99.01], [0.0, 0.09, 0.17, 0.26, 0.41, 0.65, 1.03, 1.73, 3.2, 7.09, 98.2], [0.0, 0.07, 0.13, 0.22, 0.34, 0.54, 0.89, 1.55, 2.78, 5.78, 100.0], [0.0, 0.11, 0.23, 0.4, 0.62, 0.99, 1.64, 2.71, 4.88, 9.2, 99.45], [0.0, 0.04, 0.09, 0.17, 0.3, 0.55, 0.94, 1.59, 3.0, 6.7, 83.22], [0.0, 0.06, 0.12, 0.21, 0.34, 0.54, 

In [6]:
# === Helper: Get COCO split for a given image ID ===
def find_split_for_image(img_id):
    for split, coco_obj in cocos.items():
        if img_id in coco_obj.imgs:
            return split, coco_obj
    return None, None

In [7]:
# === IoU threshold for filtering ===
IOU_THRESHOLD = 0.5

# === Data structure to store the vectors scores ===
vector_scores_struct = {cls: {b: [] for b in range(len(classes_bins[0])-1)} for cls in classes_names} 

# Loop through every class_id
for i in range(num_classes):

    # Get all image IDs for this class (from both splits)
    img_ids = []
    for split, coco_obj in cocos.items():
        img_ids += coco_obj.getImgIds(catIds=[coco_ids[i]])

    k = 0
    # Loop through each image ID
    for img_id in tqdm(img_ids, desc=f"Processing {len(img_ids)} images for {classes_names[i]}", unit="image"):
        
        # Find the split for the current image ID
        split, any_coco = find_split_for_image(img_id)
        
        # Load the image 
        img_data = any_coco.imgs[img_id]
        img_path = os.path.join(os.getenv("COCO_DATA"), split, img_data["file_name"])
        original_img = io.imread(img_path)
        
        # Check if the image is grayscale and convert to RGB if necessary
        if original_img.ndim == 2:  # grayscale
            original_img = np.stack([original_img]*3, axis=-1)


        # Augment the image
        image_set = []
        image_set.append(original_img)

        # Augmentations

        # Parse to PIL Image
        original_img = Image.fromarray(original_img)
        
        # 1 - Color Jitter
        color_jitter = T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1)
        img_color_aug = color_jitter(original_img)
        image_set.append(np.array(img_color_aug))

        # 2 - Gaussian Blur
        gaussian_blur = T.GaussianBlur(kernel_size=(7, 7), sigma=(2.5, 5.0))
        img_blur_aug = gaussian_blur(original_img)
        image_set.append(np.array(img_blur_aug))

        # 3 - Gaussian Noise
        def add_gaussian_noise(pil_img, std=25):
            arr = np.array(pil_img).astype(np.float32)
            noise = np.random.normal(0, std, arr.shape)
            noisy_arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
            return Image.fromarray(noisy_arr)

        img_noise_aug = add_gaussian_noise(original_img, std=50)
        image_set.append(np.array(img_noise_aug))

        # Load annotations for the image
        ann_ids = any_coco.getAnnIds(imgIds=img_id, catIds=[coco_ids[i]], iscrowd=None)
        anns = any_coco.loadAnns(ann_ids)

        for img_idx, img in enumerate(image_set):

            # Prediction
            results = yolo_model.predict(source=img, device='cuda:0', conf=0.25, iou=0.45, verbose=False, max_det=100)
            
            # Unpack results
            result = results[0] # One image, one result
            xyxy = result.boxes.xyxy
            xywh = result.boxes.xywh
            conf = result.boxes.conf
            cls = result.boxes.cls
            prob_vectors = result.boxes.data[:, 6:]  # Probability vectors for each class

            # # Plot the image
            # plt.figure(figsize=(5, 5))
            # plt.imshow(img)
            # plt.axis('off')
            # plt.title(f"Image ID: {img_id}, GT: {classes_names[i]}, Augmentation: {augmenting_tech[img_idx]}")

            # Loop ground truth annotations
            for ann in anns:
                bbox = ann["bbox"]
                x1_gt, y1_gt, w_gt, h_gt = bbox
                x2_gt = x1_gt + w_gt
                y2_gt = y1_gt + h_gt

                # Compute the scale of ground truth bounding box compared to whole image
                scale_gt = w_gt * h_gt / (img.shape[0] * img.shape[1])*100
                #print(f"Ground truth Scale: {scale_gt:.4f}%")

                # Compute bin index
                bin_idx = utils.bin_index(scale_gt, classes_bins[i])

                #print(f"Ground truth Bin index: {bin_idx}")
                #print("Classes bins:", classes_bins[i])

                # Plot ground truth boxes
                # plt.gca().add_patch(plt.Rectangle((x1_gt, y1_gt), w_gt, h_gt, fill=False, edgecolor="green", linewidth=2))

                # Loop through every detection
                for box_xyxy, box_xywh, c, cl, prob_v in zip(xyxy, xywh, conf, cls, prob_vectors):
                    x1, y1, x2, y2 = box_xyxy.tolist()
                    cx, cy, w, h = box_xywh.tolist()
                    confidence = c.item()
                    class_id = int(cl.item()) 
                    prob_vector = prob_v.tolist()

                    # Compute IoU
                    IoU = utils.compute_iou([x1, y1, x2, y2], [x1_gt, y1_gt, x2_gt, y2_gt])

                    # Check if IoU is above the threshold
                    if IoU > IOU_THRESHOLD:

                        # Add vector probability to the corresponding bin
                        vector_scores_struct[classes_names[i]][bin_idx].append(prob_vector)

        #                 # Print results
        #                 print(f"XYXY (corner format): ({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})")
        #                 #print(f"XYWH (center format): ({cx:.1f}, {cy:.1f}, {w:.1f}, {h:.1f})")
        #                 print(f"Confidence: {confidence:.4f}, Class ID: {class_id}, Class Name: {yolo_model.names[class_id]}")
        #                 print(f"Probability vector: {[f'{x:.3e}' for x in prob_vector]}")
        #                 print(f"Max: {max(prob_vector):.3e}, 2nd Max: {sorted(prob_vector, reverse=True)[1]:.3e}, 3rd Max: {sorted(prob_vector, reverse=True)[2]:.3e}")
        #                 print(f"Sum : {sum(prob_vector):.3e}")
        #                 #print(f"IoU: {IoU:.4f}")
        #                 print("---")

        #                 # Plot predicted boxes
        #                 plt.gca().add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", linewidth=1))
        #                 plt.text(x1, y1, f"{yolo_model.names[class_id]} {confidence:.2f}", color="red", fontsize=12)
                    
        #     plt.show()

        #     # Increment
        #     k +=1

        # if k > 5:
        #     break


Processing 2234 images for teddy bear: 100%|██████████| 2234/2234 [13:40<00:00,  2.72image/s]


In [17]:
# Save to pickle
with open("vector-scores-struct-augmented.pkl", "wb") as f:
    pickle.dump(vector_scores_struct, f)

In [19]:
# Print the number of vectors in each bin
for cls in vector_scores_struct:
    for b in vector_scores_struct[cls]:
        count = len(vector_scores_struct[cls][b]) if isinstance(vector_scores_struct[cls][b], list) else vector_scores_struct[cls][b].shape[0]
        print(f"{cls} - bin {b}: {count} vectors")


handbag - bin 0: 114 vectors
handbag - bin 1: 487 vectors
handbag - bin 2: 977 vectors
handbag - bin 3: 1593 vectors
handbag - bin 4: 2029 vectors
handbag - bin 5: 2361 vectors
handbag - bin 6: 2763 vectors
handbag - bin 7: 3187 vectors
handbag - bin 8: 3545 vectors
handbag - bin 9: 4183 vectors
bottle - bin 0: 538 vectors
bottle - bin 1: 1865 vectors
bottle - bin 2: 2887 vectors
bottle - bin 3: 4224 vectors
bottle - bin 4: 5279 vectors
bottle - bin 5: 5647 vectors
bottle - bin 6: 6668 vectors
bottle - bin 7: 7232 vectors
bottle - bin 8: 7932 vectors
bottle - bin 9: 8293 vectors
wine glass - bin 0: 174 vectors
wine glass - bin 1: 835 vectors
wine glass - bin 2: 1106 vectors
wine glass - bin 3: 1626 vectors
wine glass - bin 4: 2065 vectors
wine glass - bin 5: 2306 vectors
wine glass - bin 6: 2617 vectors
wine glass - bin 7: 2836 vectors
wine glass - bin 8: 2931 vectors
wine glass - bin 9: 2956 vectors
cup - bin 0: 507 vectors
cup - bin 1: 1865 vectors
cup - bin 2: 3472 vectors
cup - bin

In [ ]:
# Print the whole structure
print(vector_scores_struct)

{'handbag': {0: array([[   0.028782,   0.0087108,   0.0086358, ...,   0.0014594,   0.0013141,   0.0015841],
       [   0.026637,   0.0091438,    0.009564, ...,   0.0013465,   0.0013308,   0.0019016],
       [  0.0084809,    0.013077,     0.01249, ...,   0.0028016,   0.0024341,  0.00090088],
       ...,
       [   0.037466,    0.018593,    0.034762, ...,   0.0028372,   0.0020881,   0.0011265],
       [   0.028483,   0.0090323,    0.010294, ...,   0.0072075,   0.0026664,   0.0021162],
       [   0.028893,   0.0086103,    0.012683, ...,   0.0059274,   0.0025584,   0.0021418]]), 1: array([[   0.026708,    0.014069,    0.011077, ...,   0.0029997,    0.002778,   0.0022736],
       [   0.015821,   0.0089864,   0.0052877, ...,   0.0024935,   0.0027927,   0.0011777],
       [   0.019046,    0.010234,   0.0057305, ...,   0.0026125,   0.0030663,   0.0012851],
       ...,
       [   0.044815,    0.029012,   0.0065066, ...,   0.0020794,   0.0017901,   0.0007647],
       [  0.0044894,    0.010432,  